# TT-LLM: Compression + SFT + KD + Evaluation

This notebook compresses SmolLM2-360M using TT-LLM or SoLA, then:
1. **Stage 1**: Supervised Fine-Tuning (SFT) on Dolly-15K
2. **Stage 2**: Knowledge Distillation (KD) with the uncompressed teacher
3. **Evaluation**: Run 5 benchmarks (MMLU, HellaSwag, ARC-Easy, PIQA, WinoGrande)

## 0. Setup

In [ ]:
# Clone the repo and install with all Colab dependencies
import os

if not os.path.exists('/content/TT_LLM'):
    !git clone https://github.com/WaelRabah/TT_LLM.git /content/TT_LLM

%cd /content/TT_LLM
!pip install -e ".[colab]" -q

# If the repo is private, use a GitHub token instead:
# 1. Add your token as a Colab secret named 'GITHUB_TOKEN'
# 2. Uncomment the lines below:
# from google.colab import userdata
# token = userdata.get('GITHUB_TOKEN')
# !git clone https://{token}@github.com/WaelRabah/TT_LLM.git /content/TT_LLM
# %cd /content/TT_LLM
# !pip install -e ".[colab]" -q

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float32 if DEVICE == 'cpu' else torch.bfloat16

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'HuggingFaceTB/SmolLM2-360M'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 1. Load Baseline Model & Run Initial Evaluation

In [ ]:
# Load baseline (teacher) model
teacher_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map=DEVICE
)
teacher_model.eval()
baseline_params = sum(p.numel() for p in teacher_model.parameters())
print(f'Teacher model: {baseline_params:,} params')

In [ ]:
# Quick generation test
prompt = 'Explain quantum computing in one sentence.'
inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = teacher_model.generate(**inputs, max_new_tokens=50, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# Evaluate baseline on benchmarks (optional — can be slow on CPU)
# Uncomment to run:
# from tt_llm.eval import run_benchmarks, print_results
# baseline_results = run_benchmarks(teacher_model, tokenizer, device=DEVICE, limit=500)
# print_results(baseline_results, 'Baseline (no compression)')

## 2. Compress the Model

Choose one: **TT-LLM** or **SoLA**

In [ ]:
COMPRESSION_PCT = 15  # remove 15% of parameters
METHOD = 'tt_llm'  # or 'sola'

CALIB_PROMPTS = [
    'Explain the concept of quantum computing in one simple sentence.',
    'Write a short story about a robot learning to paint.',
    'What are the main causes of climate change?',
    'Describe the process of photosynthesis.',
    'How does a transformer neural network work?',
]

In [ ]:
# Load a fresh copy for the student
student_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=DTYPE, device_map=DEVICE
)

if METHOD == 'tt_llm':
    from tt_llm import compress_model_targeted
    compress_model_targeted(
        student_model, tokenizer, CALIB_PROMPTS,
        compression_pct=COMPRESSION_PCT,
        importance_cutoff=None,
        layer_type='linear',
        init_method='svd',
        max_length=128,
        device=DEVICE,
        verbose=True,
    )
elif METHOD == 'sola':
    from tt_llm import compress_model_sola
    compress_model_sola(
        student_model, tokenizer, CALIB_PROMPTS,
        compression_pct=COMPRESSION_PCT,
        sparsity_ratio=0.5,
        max_length=128,
        device=DEVICE,
        verbose=True,
    )

student_model.eval()
compressed_params = sum(p.numel() for p in student_model.parameters())
print(f'\nCompressed: {compressed_params:,} params '
      f'({baseline_params - compressed_params:,} removed, '
      f'{compressed_params/baseline_params*100:.1f}% of original)')

## 3. Stage 1: Supervised Fine-Tuning (SFT)

In [ ]:
from tt_llm.data import DollySFTDataset
from tt_llm.training import TrainConfig, train_sft

# Load Dolly-15K
train_ds = DollySFTDataset(tokenizer, split='train', max_length=512)
val_ds = DollySFTDataset(tokenizer, split='val', max_length=512)
print(f'Train: {len(train_ds)} examples')
print(f'Val: {len(val_ds)} examples')

In [ ]:
# Show a sample
sample = train_ds[0]
prompt_text = tokenizer.decode(
    [t for t, l in zip(sample['input_ids'], sample['labels']) if l == -100]
)
response_text = tokenizer.decode(
    [t for t, l in zip(sample['input_ids'], sample['labels']) if l != -100]
)
print('=== Prompt ===')
print(prompt_text[:200])
print('=== Response ===')
print(response_text[:200])

In [ ]:
# SFT config
sft_config = TrainConfig(
    output_dir='./checkpoints/sft',
    num_epochs=1,
    batch_size=8,
    gradient_accumulation_steps=4,  # effective batch = 32
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=(DEVICE == 'cuda'),
    log_every=10,
)

sft_metrics = train_sft(
    student_model, tokenizer,
    train_ds, val_ds,
    config=sft_config,
    device=DEVICE,
)

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

if sft_metrics['train_loss']:
    plt.figure(figsize=(10, 4))
    plt.plot(sft_metrics['train_loss'], alpha=0.3, label='step loss')
    # Rolling average
    window = 50
    if len(sft_metrics['train_loss']) > window:
        rolling = [sum(sft_metrics['train_loss'][max(0,i-window):i])/min(i,window) 
                   for i in range(1, len(sft_metrics['train_loss'])+1)]
        plt.plot(rolling, label=f'rolling avg ({window})', linewidth=2)
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('SFT Training Loss')
    plt.legend()
    plt.show()

if sft_metrics['val_loss']:
    print(f'Validation losses by epoch: {sft_metrics["val_loss"]}')

In [ ]:
# Test generation after SFT
student_model.eval()
inputs = tokenizer('Explain quantum computing in one sentence.', return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = student_model.generate(**inputs, max_new_tokens=50, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 4. Stage 2: Knowledge Distillation (KD)

In [ ]:
# Reload the SFT checkpoint (or continue from the model in memory)
# student_model = AutoModelForCausalLM.from_pretrained('./checkpoints/sft/final').to(DEVICE)

# Teacher should be on the same device
teacher_model.to(DEVICE)

# KD config
kd_config = TrainConfig(
    output_dir='./checkpoints/kd',
    num_epochs=1,
    batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,  # lower LR for KD
    warmup_ratio=0.1,
    weight_decay=0.01,
    bf16=(DEVICE == 'cuda'),
    kd_temperature=2.0,
    kd_alpha=0.5,  # 50% KL loss, 50% CE loss
    log_every=10,
)

kd_metrics = train_kd(
    student_model, teacher_model, tokenizer,
    train_ds, val_ds,
    config=kd_config,
    device=DEVICE,
)

In [ ]:
# Plot KD losses
if kd_metrics['train_loss']:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].plot(kd_metrics['train_loss'], alpha=0.3)
    axes[0].set_title('Total Loss')
    axes[1].plot(kd_metrics['train_kd_loss'], alpha=0.3, color='orange')
    axes[1].set_title('KD Loss (KL)')
    axes[2].plot(kd_metrics['train_ce_loss'], alpha=0.3, color='green')
    axes[2].set_title('CE Loss')
    for ax in axes:
        ax.set_xlabel('Step')
    plt.tight_layout()
    plt.show()

## 5. Final Evaluation

In [ ]:
from tt_llm.eval import run_benchmarks, print_results

# Evaluate compressed + trained model
student_model.eval()
final_results = run_benchmarks(
    student_model, tokenizer,
    device=DEVICE,
    batch_size=8,
    limit=500,  # increase or remove for full eval
)
print_results(final_results, f'{METHOD.upper()} {COMPRESSION_PCT}% (after SFT+KD)')

In [ ]:
# Also evaluate baseline for comparison (if not done earlier)
baseline_results = run_benchmarks(
    teacher_model, tokenizer,
    device=DEVICE,
    batch_size=8,
    limit=500,
)
print_results(baseline_results, 'Baseline (uncompressed)')

In [ ]:
# Comparison table
print(f'\n{"":25s}  {"Baseline":>10s}  {"Compressed":>10s}  {"Delta":>10s}')
print('-' * 60)
all_tasks = sorted(set(list(baseline_results.keys()) + list(final_results.keys())))
for task in all_tasks:
    b = baseline_results.get(task, -1)
    c = final_results.get(task, -1)
    delta = c - b if b >= 0 and c >= 0 else 0
    display = task.replace('mmlu_', 'MMLU_')
    print(f'  {display:23s}  {b:10.4f}  {c:10.4f}  {delta:+10.4f}')

print(f'\n  Params: {baseline_params:,} -> {compressed_params:,} '
      f'({compressed_params/baseline_params*100:.1f}%)')

In [ ]:
# Save final model
student_model.save_pretrained('./checkpoints/final_compressed')
tokenizer.save_pretrained('./checkpoints/final_compressed')
print('Final model saved to ./checkpoints/final_compressed')